# Городские посты: проверка утверждений как NLI

Ноутбук дообучает encoder-модель на парах `(text, claim)` и решает трёхклассовую задачу:

- `supported` — утверждение следует из поста;
- `contradicted` — пост противоречит утверждению;
- `not_stated` — в посте недостаточно сведений.

Основная модель — `cointegrated/rubert-base-cased-nli-threeway`. Её исходные метки семантически совпадают с метками соревнования. Обучение ведётся на 5 групповых фолдах, тестовые вероятности усредняются. Отдельно проверяется групповое декодирование `2/2/2` для шести утверждений одного поста.

После полного запуска создаются `artifacts/submission.csv` и `artifacts/submission_group_222.csv`. Качество здесь не задано заранее: решение о выборе файла принимается по out-of-fold macro-F1.

## 1. Подготовка окружения и модели

Для воспроизводимого запуска нужны Python 3.10/3.11, NVIDIA T4 и локально сохранённые веса. Команды установки и скачивания оставлены закомментированными: их выполняют **один раз при подготовке**, а не во время офлайн-проверки.

Если лимит 200 МБ распространяется и на веса, замените репозиторий на `cointegrated/rubert-tiny2`; для него разумно начать с learning rate `5e-5`. Не включайте тяжёлые веса в архив кода без проверки правил передачи моделей.

In [ ]:
# %pip install "torch==2.5.1" "transformers==4.48.3" "huggingface_hub==0.27.1" "safetensors==0.5.2"
# %pip install "numpy==2.0.2" "pandas==2.2.3" "scikit-learn==1.6.1"

In [ ]:
import os
from pathlib import Path

MODEL_REPO = os.getenv('NLI_MODEL_REPO', 'cointegrated/rubert-base-cased-nli-threeway')
MODEL_DIR = Path(os.getenv('NLI_MODEL_DIR', f"models/{MODEL_REPO.split('/')[-1]}"))
ALLOW_DOWNLOAD = False

if ALLOW_DOWNLOAD:
    os.environ['HF_HUB_OFFLINE'] = '0'
    os.environ['TRANSFORMERS_OFFLINE'] = '0'
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id=MODEL_REPO, local_dir=MODEL_DIR)

print('Модель:', MODEL_REPO)
print('Локальный каталог:', MODEL_DIR.resolve())

## 2. Параметры и воспроизводимость

Флаги офлайн-режима устанавливаются до импорта `transformers`. Дополнительно каждая загрузка использует `local_files_only=True`, поэтому скрытого обращения к Hugging Face во время решения не будет.

`NLI_SMOKE_TEST=1` включает только техническую проверку на небольшой выборке. Такой запуск не предназначен для отправки: он использует два фолда, одну эпоху и часть данных. Для соревнования переменная должна быть не установлена или равна `0`.

In [ ]:
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_DATASETS_OFFLINE'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTHONHASHSEED'] = '42'

import gc
import itertools
import json
import random
import re
import time
import unicodedata
from collections import Counter

import numpy as np
import pandas as pd
import torch
import transformers
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedGroupKFold
from torch.utils.data import DataLoader, Dataset
from transformers import AutoConfig, AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

SEED = 42
LABELS = ['supported', 'contradicted', 'not_stated']
SMOKE_TEST = os.getenv('NLI_SMOKE_TEST', '0') == '1'
DATA_DIR = Path(os.getenv('NLI_DATA_DIR', '.'))
OUTPUT_DIR = Path(os.getenv('NLI_OUTPUT_DIR', 'artifacts_smoke' if SMOKE_TEST else 'artifacts'))
N_FOLDS = 2 if SMOKE_TEST else 5
EPOCHS = 1 if SMOKE_TEST else 4
MAX_LENGTH = 128 if SMOKE_TEST else 512
BATCH_SIZE = 2 if SMOKE_TEST else 8
GRAD_ACCUM = 1 if SMOKE_TEST else 2
LEARNING_RATE = float(os.getenv('NLI_LEARNING_RATE', '2e-5'))
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_FP16 = DEVICE.type == 'cuda'

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print({'device': str(DEVICE), 'fp16': USE_FP16, 'smoke_test': SMOKE_TEST,
       'folds': N_FOLDS, 'epochs': EPOCHS, 'batch_size': BATCH_SIZE,
       'grad_accum': GRAD_ACCUM, 'learning_rate': LEARNING_RATE,
       'torch': torch.__version__, 'transformers': transformers.__version__})

## 3. Данные и контроль групп

Пост является группой: все его утверждения должны попасть либо в обучение фолда, либо в валидацию. Иначе почти одинаковые контексты создали бы утечку и завысили оценку.

Загрузчик сначала ищет `train.csv` и `test.csv`, а затем файлы с суффиксами вроде `train (7).csv`, которые находятся в текущей папке.

In [ ]:
def resolve_csv(stem):
    candidates = [DATA_DIR / f'{stem}.csv'] + sorted(DATA_DIR.glob(f'{stem} (*).csv'))
    candidates = [path for path in candidates if path.is_file()]
    if not candidates:
        raise FileNotFoundError(f'Не найден {stem}.csv в {DATA_DIR.resolve()}')
    return candidates[0]


def validate_raw(train_df, test_df):
    if list(train_df.columns) != ['id', 'text', 'claim', 'label']:
        raise ValueError(f'Неожиданные столбцы train: {list(train_df.columns)}')
    if list(test_df.columns) != ['id', 'text', 'claim']:
        raise ValueError(f'Неожиданные столбцы test: {list(test_df.columns)}')
    if train_df.isna().any().any() or test_df.isna().any().any():
        raise ValueError('В данных есть пропуски')
    if train_df.id.duplicated().any() or test_df.id.duplicated().any():
        raise ValueError('ID должны быть уникальны')
    unknown = set(train_df.label) - set(LABELS)
    if unknown:
        raise ValueError(f'Неизвестные метки: {unknown}')


train_raw = pd.read_csv(resolve_csv('train'))
test_raw = pd.read_csv(resolve_csv('test'))
validate_raw(train_raw, test_raw)

train = train_raw.copy()
test = test_raw.copy()

if SMOKE_TEST:
    rng = np.random.default_rng(SEED)
    required = [train_raw.loc[train_raw.label.eq(label), 'text'].iloc[0] for label in LABELS]
    selected = list(dict.fromkeys(required))
    remaining = [text for text in train_raw.text.drop_duplicates() if text not in selected]
    rng.shuffle(remaining)
    selected += remaining[:max(0, 24 - len(selected))]
    train = train_raw[train_raw.text.isin(selected)].copy().reset_index(drop=True)
    test_groups = test_raw.text.drop_duplicates().to_numpy()
    rng.shuffle(test_groups)
    test = test_raw[test_raw.text.isin(test_groups[:8])].copy().reset_index(drop=True)
    print('SMOKE TEST: файл результата не пригоден для соревнования')

print('train/test:', train.shape, test.shape)
print('Постов train/test:', train.text.nunique(), test.text.nunique())
print(train.label.value_counts().reindex(LABELS, fill_value=0))

In [ ]:
shared_texts = set(train_raw.text) & set(test_raw.text)
shared_claims = set(train_raw.claim) & set(test_raw.claim)
print('Общих постов train/test:', len(shared_texts))
print('Дословно общих утверждений:', len(shared_claims))

six_patterns = Counter()
for _, group in train_raw.groupby('text', sort=False):
    if len(group) == 6:
        counts = group.label.value_counts().reindex(LABELS, fill_value=0)
        six_patterns[tuple(counts)] += 1

test_six_groups = int((test_raw.groupby('text').size() == 6).sum())
print('Распределения классов в train-группах из 6:', dict(six_patterns))
print('Тестовых групп из 6:', test_six_groups)

## 4. Нормализация без потери важных деталей

Числа, отрицания, эмодзи и разметка не удаляются. Меняются только Unicode-представление и повторяющиеся пробелы.

Случайные идентификаторы `⟦e:...⟧` заменяются локальными токенами `<e0>`, `<e1>` и так далее. Одинаковый идентификатор внутри пары получает одинаковый токен, разные идентификаторы — разные. Так сохраняется информация о подмене участника без запоминания случайных строк.

In [ ]:
ENTITY_RE = re.compile(r'⟦e:([^⟧]+)⟧')
SPACE_RE = re.compile(r'\s+')
LOCAL_ENTITY_RE = re.compile(r'<e(\d+)>')


def normalize_pair(text, claim):
    entity_map = {}

    def convert(value):
        value = unicodedata.normalize('NFKC', value)

        def replace(match):
            entity = match.group(1)
            if entity not in entity_map:
                entity_map[entity] = f'<e{len(entity_map)}>'
            return entity_map[entity]

        value = ENTITY_RE.sub(replace, value)
        return SPACE_RE.sub(' ', value).strip()

    return convert(text), convert(claim)


def preprocess_frame(frame):
    result = frame.copy()
    pairs = [normalize_pair(text, claim) for text, claim in zip(result.text, result.claim)]
    result['text_clean'] = [pair[0] for pair in pairs]
    result['claim_clean'] = [pair[1] for pair in pairs]
    return result


train = preprocess_frame(train)
test = preprocess_frame(test)

max_entity_index = -1
for value in pd.concat([train.text_clean, train.claim_clean, test.text_clean, test.claim_clean]):
    for match in LOCAL_ENTITY_RE.finditer(value):
        max_entity_index = max(max_entity_index, int(match.group(1)))

entity_tokens = [f'<e{i}>' for i in range(max_entity_index + 1)]
print('Локальных токенов сущностей:', len(entity_tokens))
print('Пример:', train.loc[0, 'claim_clean'])

## 5. Токенизатор и порядок меток

Если у предобученной NLI-модели уже есть три осмысленные метки, их числовой порядок сохраняется. Это позволяет правильно переиспользовать обученную классификационную голову. У основной модели получится: `0=supported`, `1=contradicted`, `2=not_stated`.

Длина ограничена позиционными embeddings модели. При необходимости обрезается только первый аргумент — пост, но не утверждение. Ячейка ниже показывает, сколько пар реально затрагивает обрезание.

Для приложенных данных максимальная длина пары при токенизации основной моделью составляет 450 токенов, поэтому 512 достаточно: все посты и утверждения помещаются целиком.

In [ ]:
if not (MODEL_DIR / 'config.json').is_file():
    raise FileNotFoundError(
        f'Нет локальных весов в {MODEL_DIR.resolve()}. '
        'Скачайте модель на этапе подготовки через ALLOW_DOWNLOAD=True.'
    )

model_config = AutoConfig.from_pretrained(MODEL_DIR, local_files_only=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, local_files_only=True, use_fast=True)
if entity_tokens:
    tokenizer.add_special_tokens({'additional_special_tokens': entity_tokens})

aliases = {
    'entailment': 'supported',
    'supported': 'supported',
    'contradiction': 'contradicted',
    'contradicted': 'contradicted',
    'neutral': 'not_stated',
    'not_stated': 'not_stated',
}
raw_id2label = getattr(model_config, 'id2label', {})
candidate = {int(i): aliases.get(str(name).lower()) for i, name in raw_id2label.items()}
if set(candidate) == {0, 1, 2} and all(candidate.values()):
    ID2LABEL = dict(sorted(candidate.items()))
else:
    ID2LABEL = dict(enumerate(LABELS))
LABEL2ID = {label: i for i, label in ID2LABEL.items()}

model_positions = int(getattr(model_config, 'max_position_embeddings', 512))
MAX_LENGTH = min(MAX_LENGTH, model_positions)
print('label2id:', LABEL2ID)
print('max_length:', MAX_LENGTH, 'из', model_positions, 'позиций модели')

In [ ]:
def measure_token_lengths(frame, batch_size=128):
    lengths = []
    for start in range(0, len(frame), batch_size):
        part = frame.iloc[start:start + batch_size]
        encoded = tokenizer(
            part.text_clean.tolist(),
            part.claim_clean.tolist(),
            add_special_tokens=True,
            padding=False,
            truncation=False,
        )
        lengths.extend(len(ids) for ids in encoded['input_ids'])
    return np.asarray(lengths)


token_lengths = np.concatenate([measure_token_lengths(train), measure_token_lengths(test)])
quantiles = np.quantile(token_lengths, [0.5, 0.9, 0.95, 0.99, 1.0]).astype(int)
truncated_share = float(np.mean(token_lengths > MAX_LENGTH))
print('Длины p50/p90/p95/p99/max:', quantiles.tolist())
print(f'Пар длиннее max_length: {truncated_share:.2%}')
if truncated_share:
    print('Внимание: у части постов конец будет обрезан. Если модель позволяет, увеличьте MAX_LENGTH.')

## 6. Обучение cross-encoder

Текст и утверждение подаются в модель вместе. Это важно: модель должна сравнивать конкретные детали, а не независимо оценивать похожесть двух embeddings.

На каждом фолде модель загружается заново. Лучшая эпоха выбирается по macro-F1 валидационной части, после чего модель предсказывает test. Пять тестовых наборов вероятностей усредняются. OOF-оценка немного оптимистична из-за выбора эпохи на тех же валидационных данных, поэтому небольшие различия между настройками не стоит считать значимыми.

In [ ]:
class PairDataset(Dataset):
    def __init__(self, frame, with_labels=True):
        self.texts = frame.text_clean.tolist()
        self.claims = frame.claim_clean.tolist()
        self.labels = None
        if with_labels:
            self.labels = frame.label.map(LABEL2ID).to_numpy(dtype=np.int64)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        item = {'text': self.texts[index], 'claim': self.claims[index]}
        if self.labels is not None:
            item['label'] = int(self.labels[index])
        return item


class PairCollator:
    def __call__(self, batch):
        features = tokenizer(
            [item['text'] for item in batch],
            [item['claim'] for item in batch],
            padding=True,
            truncation='only_first',
            max_length=MAX_LENGTH,
            return_tensors='pt',
        )
        if 'label' in batch[0]:
            features['labels'] = torch.tensor([item['label'] for item in batch], dtype=torch.long)
        return features


collator = PairCollator()


def make_loader(frame, with_labels, shuffle=False, seed=SEED):
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(
        PairDataset(frame, with_labels=with_labels),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        collate_fn=collator,
        num_workers=0,
        pin_memory=DEVICE.type == 'cuda',
        generator=generator,
    )


def new_model():
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_DIR,
        num_labels=3,
        id2label=ID2LABEL,
        label2id=LABEL2ID,
        ignore_mismatched_sizes=True,
        local_files_only=True,
    )
    model.resize_token_embeddings(len(tokenizer))
    return model.to(DEVICE)


In [ ]:
@torch.no_grad()
def predict_probs(model, loader):
    model.eval()
    predictions = []
    for batch in loader:
        batch = {key: value.to(DEVICE) for key, value in batch.items() if key != 'labels'}
        with torch.cuda.amp.autocast(enabled=USE_FP16):
            logits = model(**batch).logits
        predictions.append(torch.softmax(logits.float(), dim=-1).cpu().numpy())
    return np.concatenate(predictions)


def make_optimizer(model):
    decay, no_decay = [], []
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        target = no_decay if name.endswith('bias') or 'LayerNorm' in name else decay
        target.append(parameter)
    return torch.optim.AdamW([
        {'params': decay, 'weight_decay': WEIGHT_DECAY},
        {'params': no_decay, 'weight_decay': 0.0},
    ], lr=LEARNING_RATE)


def train_epoch(model, loader, optimizer, scheduler, scaler):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    total_loss = 0.0
    for step, batch in enumerate(loader):
        batch = {key: value.to(DEVICE) for key, value in batch.items()}
        with torch.cuda.amp.autocast(enabled=USE_FP16):
            loss = model(**batch).loss
        total_loss += float(loss.detach()) * batch['labels'].shape[0]
        scaler.scale(loss / GRAD_ACCUM).backward()
        is_update = (step + 1) % GRAD_ACCUM == 0 or step + 1 == len(loader)
        if is_update:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
    return total_loss / len(loader.dataset)


## 7. Групповая кросс-валидация и ансамбль

`StratifiedGroupKFold` старается сохранить баланс меток и одновременно не разделяет посты. Веса лучших эпох сохраняются только в памяти текущего фолда; небольшие вероятности дополнительно записываются в `OUTPUT_DIR`. Для финального submission не нужен отдельный файл с весами пяти моделей.

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


y = train.label.map(LABEL2ID).to_numpy(dtype=np.int64)
cv = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
splits = list(cv.split(train, y, groups=train.text))

oof_probs = np.zeros((len(train), 3), dtype=np.float32)
test_probs = np.zeros((len(test), 3), dtype=np.float64)
fold_metrics = []
test_loader = make_loader(test, with_labels=False)
run_started = time.time()

for fold, (train_idx, valid_idx) in enumerate(splits, start=1):
    set_seed(SEED + fold)
    fold_train = train.iloc[train_idx].reset_index(drop=True)
    fold_valid = train.iloc[valid_idx].reset_index(drop=True)
    train_loader = make_loader(fold_train, with_labels=True, shuffle=True, seed=SEED + fold)
    valid_loader = make_loader(fold_valid, with_labels=True)
    model = new_model()
    optimizer = make_optimizer(model)
    updates_per_epoch = int(np.ceil(len(train_loader) / GRAD_ACCUM))
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(WARMUP_RATIO * updates_per_epoch * EPOCHS),
        num_training_steps=updates_per_epoch * EPOCHS,
    )
    scaler = torch.cuda.amp.GradScaler(enabled=USE_FP16)
    best_score = -1.0
    best_epoch = -1
    best_state = None
    best_valid_probs = None

    print(f'\nFold {fold}/{N_FOLDS}: train={len(fold_train)}, valid={len(fold_valid)}')
    for epoch in range(1, EPOCHS + 1):
        epoch_started = time.time()
        loss = train_epoch(model, train_loader, optimizer, scheduler, scaler)
        valid_probs = predict_probs(model, valid_loader)
        score = f1_score(y[valid_idx], valid_probs.argmax(axis=1), average='macro')
        print(f'  epoch={epoch}: loss={loss:.4f}, macro-F1={score:.4f}, time={time.time() - epoch_started:.1f}s')
        if score > best_score:
            best_score = score
            best_epoch = epoch
            best_valid_probs = valid_probs
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}

    model.load_state_dict(best_state)
    fold_test_probs = predict_probs(model, test_loader)
    oof_probs[valid_idx] = best_valid_probs
    test_probs += fold_test_probs / N_FOLDS
    fold_metrics.append({'fold': fold, 'best_epoch': best_epoch, 'macro_f1': float(best_score)})
    np.savez_compressed(
        OUTPUT_DIR / f'fold_{fold}_predictions.npz',
        valid_idx=valid_idx,
        valid_probs=best_valid_probs,
        test_probs=fold_test_probs,
    )

    del model, optimizer, scheduler, scaler, best_state, train_loader, valid_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

if not np.all(oof_probs.sum(axis=1) > 0):
    raise RuntimeError('OOF заполнен не полностью')

np.savez_compressed(OUTPUT_DIR / 'predictions.npz', oof_probs=oof_probs, test_probs=test_probs, y=y)
print(f'Обучение завершено за {(time.time() - run_started) / 60:.1f} мин')

## 8. Оценка по macro-F1

Смотрим не только среднее значение, но и F1 каждого класса. Матрица ошибок особенно полезна для анализа пары `contradicted` / `not_stated`: именно там легко принять отсутствие сведений за противоречие и наоборот.

In [ ]:
oof_pred = oof_probs.argmax(axis=1)
oof_f1 = f1_score(y, oof_pred, average='macro')
report_labels = list(range(3))
report_names = [ID2LABEL[i] for i in report_labels]

print(f'OOF macro-F1: {oof_f1:.5f}')
print(classification_report(y, oof_pred, labels=report_labels, target_names=report_names, digits=4, zero_division=0))
matrix = confusion_matrix(y, oof_pred, labels=report_labels)
print(pd.DataFrame(matrix, index=[f'true_{name}' for name in report_names], columns=[f'pred_{name}' for name in report_names]))
print('Лучшие эпохи:', fold_metrics)

## 9. Осторожное групповое декодирование

В полном train каждая группа из шести утверждений содержит ровно по две метки каждого класса. Для такой группы перебираются 90 допустимых распределений и выбирается максимум суммы логарифмов вероятностей. Остальные группы не изменяются.

Это не жёсткое правило для всего test. Сначала декодирование применяется к OOF: если общий macro-F1 не улучшается, основным остаётся обычный `argmax`. Для групп из пяти утверждений ограничение не применяется, поскольку там распределение не является безусловным.

In [ ]:
ASSIGNMENTS_222 = np.asarray(
    sorted(set(itertools.permutations([0, 0, 1, 1, 2, 2], 6))),
    dtype=np.int64,
)


def decode_222(probs, frame):
    result = probs.argmax(axis=1).copy()
    changed_groups = 0
    positions = np.arange(6)
    for _, indices in frame.reset_index(drop=True).groupby('text', sort=False).indices.items():
        indices = np.asarray(indices, dtype=np.int64)
        if len(indices) != 6:
            continue
        log_probs = np.log(np.clip(probs[indices], 1e-12, 1.0))
        scores = np.asarray([
            log_probs[positions, assignment].sum() for assignment in ASSIGNMENTS_222
        ])
        result[indices] = ASSIGNMENTS_222[int(scores.argmax())]
        changed_groups += 1
    return result, changed_groups


oof_pred_222, oof_decoded_groups = decode_222(oof_probs, train)
oof_f1_222 = f1_score(y, oof_pred_222, average='macro')
use_group_decode = oof_f1_222 > oof_f1

test_pred = test_probs.argmax(axis=1)
test_pred_222, test_decoded_groups = decode_222(test_probs, test)

print(f'OOF argmax: {oof_f1:.5f}')
print(f'OOF с 2/2/2: {oof_f1_222:.5f}; групп: {oof_decoded_groups}')
print(f'Тестовых групп декодировано: {test_decoded_groups}')
print('Рекомендация по OOF:', 'group_222' if use_group_decode else 'обычный argmax')

## 10. Файлы ответов

Создаются два кандидата. `submission.csv` — консервативный вариант без групповых ограничений. `submission_group_222.csv` отправляйте только при улучшении OOF. Публичный рейтинг разумно использовать как проверку, но не подгонять под него параметры: итог определяется приватной частью.

In [ ]:
def build_submission(predictions):
    return pd.DataFrame({
        'id': test.id.to_numpy(),
        'label': np.asarray([ID2LABEL[int(i)] for i in predictions]),
    })


def validate_submission(submission):
    if list(submission.columns) != ['id', 'label']:
        raise ValueError('Нужны ровно столбцы id,label')
    if len(submission) != len(test) or submission.id.duplicated().any():
        raise ValueError('Неверное число строк или повторный ID')
    if set(submission.id) != set(test.id):
        raise ValueError('Набор ID не совпадает с test')
    if not set(submission.label).issubset(set(LABELS)):
        raise ValueError('Недопустимая метка')


submission = build_submission(test_pred)
submission_222 = build_submission(test_pred_222)
validate_submission(submission)
validate_submission(submission_222)

submission.to_csv(OUTPUT_DIR / 'submission.csv', index=False)
submission_222.to_csv(OUTPUT_DIR / 'submission_group_222.csv', index=False)

for filename in ['submission.csv', 'submission_group_222.csv']:
    saved = pd.read_csv(OUTPUT_DIR / filename)
    validate_submission(saved)
    print(filename, saved.shape, saved.label.value_counts().reindex(LABELS, fill_value=0).to_dict())

metrics = {
    'model_repo': MODEL_REPO,
    'seed': SEED,
    'folds': N_FOLDS,
    'epochs': EPOCHS,
    'max_length': MAX_LENGTH,
    'learning_rate': LEARNING_RATE,
    'label2id': LABEL2ID,
    'oof_macro_f1': float(oof_f1),
    'oof_macro_f1_group_222': float(oof_f1_222),
    'use_group_decode': bool(use_group_decode),
    'fold_metrics': fold_metrics,
    'smoke_test': SMOKE_TEST,
}
with open(OUTPUT_DIR / 'metrics.json', 'w', encoding='utf-8') as file:
    json.dump(metrics, file, ensure_ascii=False, indent=2)

if SMOKE_TEST:
    print('Это технический результат на части данных, не competition submission.')
print('Файлы сохранены в', OUTPUT_DIR.resolve())

## Что проверять после полного запуска

1. В логе должно быть 5 фолдов, 2 634 тестовые строки и `smoke_test=False`.
2. Сравните macro-F1 и F1 каждого класса, а не только accuracy.
3. Если много пар длиннее 512 токенов, проверьте, не теряются ли доказательства в конце постов. Для `rubert-tiny2` можно увеличить контекст в пределах его 2048 позиций.
4. Выберите между двумя CSV по OOF, а не по случайному изменению публичного балла.
5. Перед сдачей повторите полный запуск без сети и убедитесь, что результат воспроизводится, укладывается в 4 часа на T4 и не превышает ограничения на память и архив.

Дальнейшие улучшения стоит добавлять по одному: вторая NLI-модель для ансамбля, калибровка классов или числовые/сущностные признаки. Каждое изменение нужно оценивать на тех же групповых фолдах.